# Day 094 | Automated Geotechnical Data Analysis Pipeline

## 100 Days of ML for Geotechnical Engineering
**Phase:** Geotechnical AI Projects

**Author:** Ripon Chandra Malo

---

## Project Overview

**Objective:** Build an end-to-end automated pipeline that takes raw, messy geotechnical data and produces a clean, validated, classified, and flagged dataset ready for ML or reporting.

**Pipeline Stages:**
1. **Load** raw data (messy CSV)
2. **Clean** missing values, duplicates, type errors
3. **Validate** physical plausibility (ranges, relationships)
4. **Compute** derived parameters (Vs, Dr, soil class)
5. **Classify** soil types from index properties
6. **Flag anomalies** using Isolation Forest
7. **Export** clean dataset + quality report

---

## Part 1: Imports and Setup

In [ ]:
# ─────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded.")

---

## Part 2: Generate Messy Synthetic Data

We simulate real-world data problems: missing values, outliers, typos in soil names, duplicate rows, and physically implausible entries.

In [ ]:
# ─────────────────────────────────────────────────────
# GENERATE MESSY SYNTHETIC GEOTECHNICAL DATA (100 rows)
# ─────────────────────────────────────────────────────

n = 100

# Base realistic data
borehole_ids = np.random.choice(['BH-01', 'BH-02', 'BH-03', 'BH-04'], size=n)
depths = np.round(np.random.uniform(0.5, 30.0, n), 1)
n60_values = np.random.randint(1, 60, n).astype(float)
water_content = np.random.uniform(10, 60, n)
liquid_limit = water_content + np.random.uniform(5, 30, n)
plastic_limit = water_content - np.random.uniform(0, 10, n)
fines_content = np.random.uniform(0, 100, n)
unit_weight = np.random.uniform(14, 22, n)
soil_types = np.random.choice(
    ['Sand', 'Clay', 'Silt', 'Gravel', 'Silty Sand', 'Clayey Sand'], size=n
)

df_raw = pd.DataFrame({
    'Borehole': borehole_ids,
    'Depth_m': depths,
    'N60': n60_values,
    'Water_Content_pct': np.round(water_content, 1),
    'Liquid_Limit': np.round(liquid_limit, 1),
    'Plastic_Limit': np.round(plastic_limit, 1),
    'Fines_pct': np.round(fines_content, 1),
    'Unit_Weight_kN_m3': np.round(unit_weight, 2),
    'Soil_Type': soil_types,
})

# ── INJECT MESSINESS ──

# 1. Missing values (random)
for col in ['N60', 'Water_Content_pct', 'Liquid_Limit', 'Plastic_Limit', 'Unit_Weight_kN_m3']:
    mask = np.random.random(n) < 0.08
    df_raw.loc[mask, col] = np.nan

# 2. Duplicates
dup_rows = df_raw.sample(5, random_state=10)
df_raw = pd.concat([df_raw, dup_rows], ignore_index=True)

# 3. Typos in soil names
typo_idx = np.random.choice(len(df_raw), 6, replace=False)
typos = ['SAND', 'clay', 'Snad', 'Caly', 'silt', 'gravl']
for i, idx in enumerate(typo_idx):
    df_raw.loc[idx, 'Soil_Type'] = typos[i]

# 4. Physically implausible values
df_raw.loc[3, 'N60'] = -5          # Negative N60
df_raw.loc[7, 'Depth_m'] = -2.0    # Negative depth
df_raw.loc[15, 'Unit_Weight_kN_m3'] = 35.0  # Too heavy
df_raw.loc[22, 'Water_Content_pct'] = 200.0  # Impossible water content
df_raw.loc[30, 'Fines_pct'] = 120.0          # Over 100%

print(f"Raw data: {len(df_raw)} rows, {df_raw.shape[1]} columns")
print(f"Missing values: {df_raw.isnull().sum().sum()}")
df_raw.head(10)

In [ ]:
# ─────────────────────────────────────────────────────
# RAW DATA QUALITY SNAPSHOT
# ─────────────────────────────────────────────────────

print("RAW DATA QUALITY REPORT")
print("=" * 50)
print(f"Total rows:      {len(df_raw)}")
print(f"Duplicate rows:  {df_raw.duplicated().sum()}")
print(f"\nMissing values per column:")
for col in df_raw.columns:
    missing = df_raw[col].isnull().sum()
    if missing > 0:
        print(f"  {col:25s}: {missing:3d} ({100*missing/len(df_raw):.1f}%)")
print(f"\nUnique soil types (including typos): {df_raw['Soil_Type'].nunique()}")
print(df_raw['Soil_Type'].value_counts())

---

## Part 3: Pipeline Stage 1 -- Clean

Remove duplicates, fix typos, handle missing values.

In [ ]:
# ─────────────────────────────────────────────────────
# STAGE 1: CLEAN
# ─────────────────────────────────────────────────────

pipeline_log = []  # Track all pipeline actions

def log_action(stage, action, detail):
    pipeline_log.append({'Stage': stage, 'Action': action, 'Detail': detail})
    print(f"  [{stage}] {action}: {detail}")

df = df_raw.copy()
print("STAGE 1: CLEANING")
print("-" * 40)

# 1a. Remove duplicates
n_dups = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
log_action('Clean', 'Remove duplicates', f'{n_dups} rows removed')

# 1b. Fix soil type typos
soil_map = {
    'SAND': 'Sand', 'Snad': 'Sand', 'sand': 'Sand',
    'clay': 'Clay', 'Caly': 'Clay', 'CLAY': 'Clay',
    'silt': 'Silt', 'SILT': 'Silt',
    'gravl': 'Gravel', 'GRAVEL': 'Gravel',
}
n_typos = df['Soil_Type'].isin(soil_map.keys()).sum()
df['Soil_Type'] = df['Soil_Type'].replace(soil_map)
log_action('Clean', 'Fix soil typos', f'{n_typos} entries corrected')

# 1c. Handle missing values -- group median then global median
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    n_miss = df[col].isnull().sum()
    if n_miss > 0:
        df[col] = df.groupby('Borehole')[col].transform(
            lambda x: x.fillna(x.median())
        )
        remaining = df[col].isnull().sum()
        if remaining > 0:
            df[col].fillna(df[col].median(), inplace=True)
        log_action('Clean', f'Fill missing {col}', f'{n_miss} values imputed')

print(f"\nAfter cleaning: {len(df)} rows, {df.isnull().sum().sum()} missing")

---

## Part 4: Pipeline Stage 2 -- Validate

Check physical plausibility: depth > 0, 0 < N60 < 100, unit weight 10-28 kN/m3, etc.

In [ ]:
# ─────────────────────────────────────────────────────
# STAGE 2: VALIDATE
# ─────────────────────────────────────────────────────

print("\nSTAGE 2: VALIDATION")
print("-" * 40)

validation_rules = {
    'Depth_m': (0.0, 100.0),
    'N60': (0.0, 100.0),
    'Water_Content_pct': (0.0, 150.0),
    'Liquid_Limit': (0.0, 200.0),
    'Plastic_Limit': (0.0, 100.0),
    'Fines_pct': (0.0, 100.0),
    'Unit_Weight_kN_m3': (10.0, 28.0),
}

df['Validation_Flags'] = ''

for col, (low, high) in validation_rules.items():
    invalid = (df[col] < low) | (df[col] > high)
    n_invalid = invalid.sum()
    if n_invalid > 0:
        df.loc[invalid, 'Validation_Flags'] += f'{col}_OOB; '
        df.loc[invalid, col] = df[col].clip(low, high)
        log_action('Validate', f'{col} range check', 
                   f'{n_invalid} values clipped to [{low}, {high}]')

# Check PI consistency: LL should be >= PL
pi_bad = df['Liquid_Limit'] < df['Plastic_Limit']
if pi_bad.sum() > 0:
    df.loc[pi_bad, 'Validation_Flags'] += 'PI_inconsistent; '
    log_action('Validate', 'PI consistency', f'{pi_bad.sum()} rows with LL < PL')

n_flagged = (df['Validation_Flags'] != '').sum()
print(f"\nRows with validation flags: {n_flagged}")

---

## Part 5: Pipeline Stage 3 -- Compute Derived Parameters

In [ ]:
# ─────────────────────────────────────────────────────
# STAGE 3: COMPUTE DERIVED PARAMETERS
# ─────────────────────────────────────────────────────

print("\nSTAGE 3: COMPUTE")
print("-" * 40)

# Shear wave velocity from N60 (Imai & Tonouchi, 1982)
df['Vs_m_s'] = np.round(97 * (df['N60'] ** 0.314), 1)
log_action('Compute', 'Vs from N60', 'Imai & Tonouchi (1982): Vs = 97 * N60^0.314')

# Plasticity Index
df['PI'] = np.round(df['Liquid_Limit'] - df['Plastic_Limit'], 1)
df['PI'] = df['PI'].clip(lower=0)
log_action('Compute', 'Plasticity Index', 'PI = LL - PL (clipped >= 0)')

# Relative density estimate (Meyerhof, 1957)
df['Dr_pct'] = np.round(np.sqrt(df['N60'] / 0.46) * 10, 1)
df['Dr_pct'] = df['Dr_pct'].clip(0, 100)
log_action('Compute', 'Relative Density', 'Dr = sqrt(N60/0.46)*10 (for sands)')

# Overburden stress (simplified, no pore pressure)
df['Sigma_v_kPa'] = np.round(df['Unit_Weight_kN_m3'] * df['Depth_m'], 1)
log_action('Compute', 'Overburden stress', 'sigma_v = gamma * depth')

# SPT consistency/density class
conditions = [
    df['N60'] < 4,
    (df['N60'] >= 4) & (df['N60'] < 10),
    (df['N60'] >= 10) & (df['N60'] < 30),
    (df['N60'] >= 30) & (df['N60'] < 50),
    df['N60'] >= 50
]
choices = ['Very Loose', 'Loose', 'Medium Dense', 'Dense', 'Very Dense']
df['Density_Class'] = np.select(conditions, choices, default='Unknown')
log_action('Compute', 'Density class', 'Based on N60 ranges')

print(f"\nNew columns: Vs_m_s, PI, Dr_pct, Sigma_v_kPa, Density_Class")
df[['Depth_m', 'N60', 'Vs_m_s', 'PI', 'Dr_pct', 'Sigma_v_kPa', 'Density_Class']].head(10)

---

## Part 6: Pipeline Stage 4 -- Classify Soils

In [ ]:
# ─────────────────────────────────────────────────────
# STAGE 4: CLASSIFY SOIL (simplified USCS-like)
# ─────────────────────────────────────────────────────

print("\nSTAGE 4: CLASSIFY")
print("-" * 40)

def classify_soil(row):
    """Simplified USCS classification from index properties."""
    fines = row['Fines_pct']
    pi = row['PI']
    ll = row['Liquid_Limit']
    
    if fines < 5:
        return 'SP'  # Poorly graded sand/gravel
    elif fines < 12:
        if pi < 4:
            return 'SM'  # Silty sand
        else:
            return 'SC'  # Clayey sand
    elif fines < 50:
        if pi < 7:
            return 'SM'  # Silty sand
        elif pi < 15:
            return 'SC'  # Clayey sand
        else:
            return 'CL'  # Low plasticity clay
    else:
        if pi < 10:
            return 'ML'  # Low plasticity silt
        elif ll < 50:
            return 'CL'  # Low plasticity clay
        elif pi < 25:
            return 'MH'  # High plasticity silt
        else:
            return 'CH'  # High plasticity clay

df['USCS_Auto'] = df.apply(classify_soil, axis=1)
log_action('Classify', 'Auto-USCS', 'Based on fines, PI, LL')

print(f"\nAutomatic USCS classification:")
print(df['USCS_Auto'].value_counts())

---

## Part 7: Pipeline Stage 5 -- Anomaly Detection

In [ ]:
# ─────────────────────────────────────────────────────
# STAGE 5: FLAG ANOMALIES (Isolation Forest)
# ─────────────────────────────────────────────────────

print("\nSTAGE 5: ANOMALY DETECTION")
print("-" * 40)

anomaly_features = ['Depth_m', 'N60', 'Water_Content_pct', 
                     'Unit_Weight_kN_m3', 'Fines_pct']

iso_forest = IsolationForest(contamination=0.05, random_state=42)
df['Anomaly'] = iso_forest.fit_predict(df[anomaly_features])
df['Anomaly_Flag'] = df['Anomaly'].map({1: 'Normal', -1: 'Anomaly'})

n_anomalies = (df['Anomaly'] == -1).sum()
log_action('Anomaly', 'Isolation Forest', f'{n_anomalies} anomalies detected')

print(f"\nAnomaly summary:")
print(df['Anomaly_Flag'].value_counts())

if n_anomalies > 0:
    print(f"\nAnomalous records:")
    anom_cols = ['Borehole', 'Depth_m', 'N60', 'Water_Content_pct', 
                'Unit_Weight_kN_m3', 'Soil_Type']
    print(df[df['Anomaly'] == -1][anom_cols].to_string())

In [ ]:
# ─────────────────────────────────────────────────────
# VISUALIZE ANOMALIES
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

normal = df[df['Anomaly'] == 1]
anomalies = df[df['Anomaly'] == -1]

pairs = [('Depth_m', 'N60'), ('N60', 'Water_Content_pct'), 
         ('Fines_pct', 'Unit_Weight_kN_m3')]

for ax, (x, y) in zip(axes, pairs):
    ax.scatter(normal[x], normal[y], c='#2196F3', s=20, alpha=0.5, label='Normal')
    ax.scatter(anomalies[x], anomalies[y], c='red', s=80, marker='X', 
               zorder=5, label='Anomaly')
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Anomaly Detection Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 8: Pipeline Stage 6 -- Export and Quality Report

In [ ]:
# ─────────────────────────────────────────────────────
# STAGE 6: EXPORT CLEAN DATASET
# ─────────────────────────────────────────────────────

print("\nSTAGE 6: EXPORT")
print("-" * 40)

export_cols = ['Borehole', 'Depth_m', 'N60', 'Vs_m_s', 'Water_Content_pct',
               'Liquid_Limit', 'Plastic_Limit', 'PI', 'Fines_pct',
               'Unit_Weight_kN_m3', 'Sigma_v_kPa', 'Dr_pct',
               'Soil_Type', 'USCS_Auto', 'Density_Class', 'Anomaly_Flag']

df_clean = df[export_cols].copy()
df_clean = df_clean.sort_values(['Borehole', 'Depth_m']).reset_index(drop=True)

print(f"Clean dataset: {len(df_clean)} rows, {len(export_cols)} columns")
print(f"Missing values: {df_clean.isnull().sum().sum()}")
df_clean.head(10)

In [ ]:
# ─────────────────────────────────────────────────────
# QUALITY REPORT
# ─────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("  DATA QUALITY PIPELINE REPORT")
print("=" * 60)
print(f"  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Input rows:  {len(df_raw)}")
print(f"  Output rows: {len(df_clean)}")
print(f"  Columns:     {len(export_cols)}")
print()
print("  Pipeline Actions:")
print("  " + "-" * 56)
for entry in pipeline_log:
    print(f"  [{entry['Stage']:10s}] {entry['Action']:25s} | {entry['Detail']}")
print("  " + "-" * 56)

print(f"\n  Boreholes:         {df_clean['Borehole'].nunique()}")
print(f"  Depth range:       {df_clean['Depth_m'].min():.1f} - {df_clean['Depth_m'].max():.1f} m")
print(f"  N60 range:         {df_clean['N60'].min():.0f} - {df_clean['N60'].max():.0f}")
print(f"  USCS classes:      {df_clean['USCS_Auto'].nunique()}")
print(f"  Anomalies flagged: {(df_clean['Anomaly_Flag']=='Anomaly').sum()}")
print("=" * 60)

---

## Part 9: Visualization Summary

In [ ]:
# ─────────────────────────────────────────────────────
# DASHBOARD-STYLE SUMMARY
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. N60 with depth per borehole
for bh in sorted(df_clean['Borehole'].unique()):
    mask = df_clean['Borehole'] == bh
    axes[0, 0].scatter(df_clean.loc[mask, 'N60'], df_clean.loc[mask, 'Depth_m'],
                       s=20, alpha=0.6, label=bh)
axes[0, 0].invert_yaxis()
axes[0, 0].set_xlabel('N60')
axes[0, 0].set_ylabel('Depth (m)')
axes[0, 0].set_title('N60 vs Depth', fontweight='bold')
axes[0, 0].legend(fontsize=7)
axes[0, 0].grid(True, alpha=0.3)

# 2. USCS distribution
df_clean['USCS_Auto'].value_counts().plot(kind='bar', ax=axes[0, 1],
    color='#2196F3', edgecolor='white')
axes[0, 1].set_title('USCS Classification', fontweight='bold')
axes[0, 1].set_ylabel('Count')

# 3. Density class
order = ['Very Loose', 'Loose', 'Medium Dense', 'Dense', 'Very Dense']
density_counts = df_clean['Density_Class'].value_counts().reindex(order).dropna()
density_counts.plot(kind='bar', ax=axes[0, 2], color='#FF9800', edgecolor='white')
axes[0, 2].set_title('Density Classification', fontweight='bold')
axes[0, 2].set_ylabel('Count')

# 4. Vs histogram
axes[1, 0].hist(df_clean['Vs_m_s'], bins=20, color='#4CAF50', edgecolor='white')
axes[1, 0].set_xlabel('Vs (m/s)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Shear Wave Velocity', fontweight='bold')

# 5. Plasticity chart
axes[1, 1].scatter(df_clean['Liquid_Limit'], df_clean['PI'], 
                   c='#9C27B0', s=20, alpha=0.6)
ll_line = np.linspace(0, 100, 50)
axes[1, 1].plot(ll_line, 0.73 * (ll_line - 20), 'k--', alpha=0.5, label='A-line')
axes[1, 1].set_xlabel('Liquid Limit')
axes[1, 1].set_ylabel('Plasticity Index')
axes[1, 1].set_title('Plasticity Chart', fontweight='bold')
axes[1, 1].legend()
axes[1, 1].set_xlim(0, 100)
axes[1, 1].set_ylim(0, 60)
axes[1, 1].grid(True, alpha=0.3)

# 6. Unit weight vs fines
axes[1, 2].scatter(df_clean['Fines_pct'], df_clean['Unit_Weight_kN_m3'],
                   c='#F44336', s=20, alpha=0.6)
axes[1, 2].set_xlabel('Fines Content (%)')
axes[1, 2].set_ylabel('Unit Weight (kN/m3)')
axes[1, 2].set_title('Unit Weight vs Fines', fontweight='bold')
axes[1, 2].grid(True, alpha=0.3)

fig.suptitle('Processed Geotechnical Data Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 10: Reusable Pipeline Class

In [ ]:
# ─────────────────────────────────────────────────────
# REUSABLE PIPELINE CLASS
# ─────────────────────────────────────────────────────

class GeotechDataPipeline:
    """Automated geotechnical data analysis pipeline.
    
    Stages: load -> clean -> validate -> compute -> classify -> detect_anomalies
    
    Usage:
        pipe = GeotechDataPipeline()
        result = (
            pipe.load(df_raw)
                .clean(soil_map={'SAND': 'Sand'})
                .validate()
                .compute()
                .detect_anomalies(['Depth_m', 'N60'])
                .report()
        )
    """
    
    def __init__(self):
        self.log = []
        self.df = None
        
    def _log(self, stage, msg):
        self.log.append({'stage': stage, 'message': msg})
        
    def load(self, df):
        self.df = df.copy()
        self._log('load', f'Loaded {len(self.df)} rows, {self.df.shape[1]} cols')
        return self
    
    def clean(self, soil_map=None):
        n_dup = self.df.duplicated().sum()
        self.df = self.df.drop_duplicates().reset_index(drop=True)
        self._log('clean', f'Removed {n_dup} duplicates')
        
        if soil_map and 'Soil_Type' in self.df.columns:
            self.df['Soil_Type'] = self.df['Soil_Type'].replace(soil_map)
            self._log('clean', 'Applied soil name corrections')
        
        for col in self.df.select_dtypes(include=[np.number]).columns:
            n_miss = self.df[col].isnull().sum()
            if n_miss > 0:
                self.df[col].fillna(self.df[col].median(), inplace=True)
                self._log('clean', f'Imputed {n_miss} missing in {col}')
        return self
    
    def validate(self, rules=None):
        if rules is None:
            rules = {'Depth_m': (0, 100), 'N60': (0, 100), 'Fines_pct': (0, 100)}
        for col, (lo, hi) in rules.items():
            if col in self.df.columns:
                bad = ((self.df[col] < lo) | (self.df[col] > hi)).sum()
                self.df[col] = self.df[col].clip(lo, hi)
                if bad > 0:
                    self._log('validate', f'Clipped {bad} values in {col}')
        return self
    
    def compute(self):
        if 'N60' in self.df.columns:
            self.df['Vs_m_s'] = np.round(97 * (self.df['N60'] ** 0.314), 1)
            self._log('compute', 'Added Vs_m_s')
        if {'Liquid_Limit', 'Plastic_Limit'}.issubset(self.df.columns):
            self.df['PI'] = (self.df['Liquid_Limit'] - self.df['Plastic_Limit']).clip(lower=0)
            self._log('compute', 'Added PI')
        return self
    
    def detect_anomalies(self, features, contamination=0.05):
        iso = IsolationForest(contamination=contamination, random_state=42)
        valid_feats = [f for f in features if f in self.df.columns]
        self.df['Anomaly'] = iso.fit_predict(self.df[valid_feats])
        n_anom = (self.df['Anomaly'] == -1).sum()
        self._log('anomaly', f'Detected {n_anom} anomalies')
        return self
    
    def report(self):
        print("Pipeline Report")
        print("=" * 50)
        for entry in self.log:
            print(f"  [{entry['stage']:10s}] {entry['message']}")
        print(f"  Final shape: {self.df.shape}")
        return self.df

# Run the reusable pipeline
pipe = GeotechDataPipeline()
result = (
    pipe.load(df_raw)
        .clean(soil_map=soil_map)
        .validate()
        .compute()
        .detect_anomalies(['Depth_m', 'N60', 'Water_Content_pct'])
        .report()
)
print(f"\nPipeline complete. Output shape: {result.shape}")

---

## Key Takeaways

1. **Automated pipelines save hours** of manual data cleaning. A six-stage pipeline (load, clean, validate, compute, classify, flag) handles the repetitive work every geotechnical project requires.

2. **Data validation with physical rules** catches errors that statistical methods miss. An N60 of -5 or a unit weight of 35 kN/m3 are physically impossible.

3. **Isolation Forest detects multivariate anomalies** that simple range checks miss -- e.g., a high N60 at very shallow depth, or high water content with low fines.

4. **Logging every pipeline action** creates an audit trail. When a regulator or reviewer asks what happened to the data, you have a complete record.

5. **Wrapping the pipeline in a class** makes it reusable across projects. Load new data, run the same pipeline, get consistent results.

---

## Next Steps

- Day 095: Reduced-order modeling for geotechnics
- Add database integration (SQLite, PostgreSQL)
- Build Airflow/Prefect DAG for scheduled pipeline runs

---

*Day 094 of 100 -- Clean data is the foundation of good engineering!*